In [ ]:
import json
import re
from pathlib import Path
from datetime import date

# Этап 1: txt-файлы с дашбордов -> JSON

Минимальный вариант: только чтение и парсинг, без Excel и без справочника.

1. Ищет все `.txt`-файлы рекурсивно внутри папки `txt/` (в любой вложенной подпапке — не привязано к дате).
2. Дэш и Экран определяются из **имени файла** (`<Дэш>_<Экран>.txt`).
3. Каждый файл разбирается на виджеты (показатели) тем же парсером, что и раньше.
4. Результат сохраняется в один `combined_widgets.json` — его на следующем этапе будем отправлять в LLM.

**Зависимости не нужны вообще** — только стандартная библиотека Python.

In [ ]:
MONTHS_RU = {
    "янв": 1, "января": 1, "январь": 1,
    "фев": 2, "февраля": 2, "февраль": 2,
    "мар": 3, "марта": 3, "март": 3,
    "апр": 4, "апреля": 4, "апрель": 4,
    "май": 5, "мая": 5,
    "июн": 6, "июня": 6, "июнь": 6,
    "июл": 7, "июля": 7, "июль": 7,
    "авг": 8, "августа": 8, "август": 8,
    "сен": 9, "сентября": 9, "сентябрь": 9,
    "окт": 10, "октября": 10, "октябрь": 10,
    "ноя": 11, "ноября": 11, "ноябрь": 11,
    "дек": 12, "декабря": 12, "декабрь": 12,
}

# месяц с явным годом: "авг2025", "янв.24", "январь 2024", "01.2024", "2024-01"
MONTH_WITH_YEAR_RE = re.compile(
    r"^(?:"
    r"(?P<name>[а-яё]+)\.?\s*['`]?(?P<y1>\d{2,4})"
    r"|(?P<mm>\d{1,2})[./-](?P<y2>\d{2,4})"
    r"|(?P<y3>\d{4})[./-](?P<mm2>\d{1,2})"
    r")$",
    re.IGNORECASE,
)
BARE_MONTH_RE = re.compile(r"^[а-яё]+$", re.IGNORECASE)

# квартал: "3Q2025", "4Q", "1кв2026", "2 кв. 25"
QUARTER_RE = re.compile(
    r"^(?P<num>[1-4])\s*(?:q|кв)\.?\s*(?P<year>\d{2,4})?$",
    re.IGNORECASE,
)

NUMBER_RE = re.compile(r"^-?\d[\d\s.,]*%?$")
HAS_DIGIT_RE = re.compile(r"\d")  # используется для проверки недельного среза - хватает одной цифры

FORECAST_KEYWORDS = ("прогноз",)
WEEKLY_KEYWORDS = ("нед",)

# единицы измерения (руб может быть сокращён до одной буквы "Р")
UNIT_PATTERNS = [
    re.compile(r"^(млн|тыс|млрд|трлн)\.?\s*(руб|р)\.?$", re.IGNORECASE),
    re.compile(r"^(руб|р)\.?$", re.IGNORECASE),
    re.compile(r"^%$"),
    re.compile(r"^(млн|тыс|млрд|трлн)?\.?\s*шт\.?$", re.IGNORECASE),
    re.compile(r"^(млн|тыс|млрд|трлн)?\.?\s*ед\.?$", re.IGNORECASE),
    # добавьте сюда свои варианты единиц измерения, если парсер их не находит
]

# служебные слова-подписи (легенда/KPI-плашки) - сигнализируют, что заголовок только что закончился
LABEL_STEMS = ("прогноз", "план", "факт", "выполнение", "вып", "дельта")


def is_unit_line(token: str) -> bool:
    t = token.strip()
    return any(p.match(t) for p in UNIT_PATTERNS)


def _normalize_label(token: str) -> str:
    t = token.strip().lower()
    t = re.sub(r"[.\-):]+$", "", t)
    t = re.sub(r"^[.\-(:]+", "", t)
    return t


def is_label_or_unit_line(token: str) -> bool:
    if is_unit_line(token):
        return True
    t = _normalize_label(token)
    return any(t == stem or t.startswith(stem) for stem in LABEL_STEMS)


def _month_name_to_num(name: str):
    name = name.lower()
    for key, num in MONTHS_RU.items():
        if name.startswith(key):
            return num
    return None


def parse_period_token(token: str, current_year):
    '''
    Пытается распознать токен как месяц или квартал (с явным годом или без - тогда
    используется "протянутый" current_year). Возвращает ((тип, год, номер), новый_год)
    либо (None, current_year). тип = "M" (месяц) или "Q" (квартал).
    '''
    t = token.strip().lower()

    m = MONTH_WITH_YEAR_RE.match(t)
    if m:
        if m.group("name"):
            month = _month_name_to_num(m.group("name"))
            year = m.group("y1")
        elif m.group("mm"):
            month = int(m.group("mm"))
            year = m.group("y2")
        else:
            month = int(m.group("mm2"))
            year = m.group("y3")
        if month is not None and 1 <= month <= 12:
            year = int(year)
            if year < 100:
                year += 2000
            return ("M", year, month), year

    q = QUARTER_RE.match(t)
    if q:
        num = int(q.group("num"))
        year = q.group("year")
        if year is not None:
            year = int(year)
            if year < 100:
                year += 2000
            return ("Q", year, num), year
        elif current_year is not None:
            return ("Q", current_year, num), current_year

    if BARE_MONTH_RE.match(t) and current_year is not None:
        month = _month_name_to_num(t)
        if month is not None:
            return ("M", current_year, month), current_year

    return None, current_year


def split_line(line: str):
    if "\t" in line:
        parts = line.split("\t")
    else:
        parts = re.split(r"\s{2,}", line)
    return [p.strip() for p in parts if p.strip() != ""]


def to_number(s: str):
    s = s.replace("\xa0", "").replace(" ", "").replace("%", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def flatten_tokens(text: str):
    tokens = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        tokens.extend(split_line(line))
    return tokens


def format_period(p):
    ptype, year, num = p
    if ptype == "M":
        return f"{num:02d}.{year}"
    return f"{num}кв.{year}"


def period_sort_key(p):
    return (p[1], p[2])

## Парсер

Правила:

- **Заголовок первого виджета в файле:**
  - для Дэшей, чьё имя содержит **"CB"** или **"core"** — заголовок это строка перед **первой строкой с цифрой**;
  - для Дэшей с **"УБ"** и всех остальных — строка перед первым служебным словом (единица измерения / Прогноз / План / Факт / Вып / Дельта).
- **Заголовки последующих виджетов** — токен сразу после конца числового ряда предыдущего виджета.
- **Выравнивание плана**: по умолчанию по последним датам; для Дэшей с "УБ" — по первым.
- **Недельный срез**: есть, если после "нед" идёт строка хотя бы с одной цифрой (не прочерк).

In [ ]:
def parse_widgets(text: str, dash_name: str = ""):
    '''
    Разбирает текст, скопированный с экрана, на отдельные виджеты (показатели).
    Возвращает список словарей:
      {"title", "forecast", "weekly", "periods": [(тип, год, номер), ...], "fact": [...], "plan": [...]}

    dash_name используется только для выбора правила определения ПЕРВОГО заголовка:
      - если в dash_name есть "CB" или "core" - заголовок это строка перед первой строкой с цифрой;
      - иначе (в т.ч. "УБ") - строка перед первым служебным словом (единица/Прогноз/План/Факт/Вып/Дельта).
    '''
    tokens = flatten_tokens(text)
    n = len(tokens)
    widgets = []

    use_digit_rule = ("CB" in dash_name) or ("core" in dash_name)

    pending_meta = []
    current = None
    current_year = None
    awaiting_title = True
    first_widget = True

    def new_widget(title):
        return {"title": title, "forecast": False, "weekly": False, "unit": None,
                "periods": [], "fact": [], "plan": []}

    def check_forecast(widget, token):
        if widget is not None and "прогноз" in token.lower():
            widget["forecast"] = True

    def check_weekly(widget, idx):
        if widget is None:
            return
        tok = tokens[idx]
        if "нед" in tok.lower():
            nxt = tokens[idx + 1] if idx + 1 < n else None
            if nxt is not None and HAS_DIGIT_RE.search(nxt):
                widget["weekly"] = True

    def is_first_title_trigger(token: str) -> bool:
        if use_digit_rule:
            return bool(HAS_DIGIT_RE.search(token))
        return is_label_or_unit_line(token)

    i = 0
    while i < n:
        tok = tokens[i]

        if awaiting_title:
            if first_widget:
                if is_first_title_trigger(tok) and pending_meta:
                    if current and current["periods"]:
                        widgets.append(current)
                    current = new_widget(pending_meta[-1])
                    current_year = None
                    if is_unit_line(tok):
                        current["unit"] = tok
                    check_forecast(current, tok)
                    check_weekly(current, i)
                    pending_meta = []
                    awaiting_title = False
                    first_widget = False
                    if use_digit_rule:
                        # правило CB/core: триггерный токен (первая строка с цифрой) - это
                        # и есть начало периода, его нельзя "съедать", обрабатываем заново
                        continue
                    i += 1
                    continue
            else:
                if current and current["periods"]:
                    widgets.append(current)
                current = new_widget(tok)
                current_year = None
                if is_unit_line(tok):
                    current["unit"] = tok
                check_forecast(current, tok)
                check_weekly(current, i)
                awaiting_title = False
                i += 1
                continue

        parsed, y2 = (None, current_year)
        if current is not None:
            parsed, y2 = parse_period_token(tok, current_year)

        if parsed is not None:
            periods_buf = [parsed]
            current_year = y2
            i += 1
            while i < n:
                p2, y3 = parse_period_token(tokens[i], current_year)
                if p2 is None or p2[0] != parsed[0]:
                    break
                periods_buf.append(p2)
                current_year = y3
                i += 1
            current["periods"] = periods_buf

            def consume_numbers(limit):
                nonlocal i
                vals = []
                while i < n and len(vals) < limit and NUMBER_RE.match(tokens[i]):
                    vals.append(to_number(tokens[i]))
                    i += 1
                return vals

            current["fact"] = consume_numbers(len(periods_buf))
            current["plan"] = consume_numbers(len(periods_buf))
            awaiting_title = True
            continue

        check_forecast(current, tok)
        check_weekly(current, i)
        if current is not None and current["unit"] is None and is_unit_line(tok):
            current["unit"] = tok
        pending_meta.append(tok)
        i += 1

    if current and current["periods"]:
        widgets.append(current)

    return widgets

## Сбор всех файлов и выгрузка в JSON

In [ ]:
TXT_ROOT = Path("txt")
OUTPUT_JSON = Path("combined_widgets.json")


def format_period(p):
    ptype, year, num = p
    if ptype == "M":
        return f"{num:02d}.{year}"
    return f"{num}кв.{year}"


def compute_actual_period(periods, fact, today=None):
    today = today or date.today()
    today_q = (today.month - 1) // 3 + 1
    dated = [p for p, val in zip(periods, fact) if val is not None]
    if not dated:
        return None

    def is_past_or_present(p):
        ptype, year, num = p
        if ptype == "M":
            return (year, num) <= (today.year, today.month)
        return (year, num) <= (today.year, today_q)

    past = [p for p in dated if is_past_or_present(p)]
    pool = past if past else dated
    return max(pool, key=lambda p: (p[1], p[2]))


def align_plan_to_periods(periods, plan, is_ub_dash: bool):
    if not plan:
        return {}
    if is_ub_dash:
        plan_periods = periods[:len(plan)]
    else:
        plan_periods = periods[len(periods) - len(plan):]
    return dict(zip(plan_periods, plan))


if not TXT_ROOT.exists():
    raise FileNotFoundError(f"Папка {TXT_ROOT.resolve()} не найдена")

txt_files = sorted(TXT_ROOT.glob("**/*.txt"))
print(f"Найдено txt-файлов: {len(txt_files)}")

result = []

for fpath in txt_files:
    stem = fpath.stem
    dash, _, screen = stem.partition("_")
    is_ub_dash = "УБ" in dash

    text = fpath.read_text(encoding="utf-8")
    widgets = parse_widgets(text, dash_name=dash)

    if not widgets:
        print(f"⚠️ Ни одного виджета не распознано в {fpath} - проверьте формат файла")

    widgets_out = []
    for w in widgets:
        periods = w["periods"]
        fact = w["fact"]
        plan = w["plan"]

        if not periods or not fact:
            continue

        plan_dict = align_plan_to_periods(periods, plan, is_ub_dash)
        actual_period = compute_actual_period(periods, fact)

        values = []
        for idx, p in enumerate(periods):
            values.append({
                "period": format_period(p),
                "fact": fact[idx] if idx < len(fact) else None,
                "plan": plan_dict.get(p),
            })

        widgets_out.append({
            "title": w["title"],
            "unit": w.get("unit"),
            "granularity": "Месяц" if periods[0][0] == "M" else "Квартал",
            "forecast": w["forecast"],
            "weekly": w["weekly"],
            "actual_period": format_period(actual_period) if actual_period else None,
            "values": values,
        })

    result.append({
        "dash": dash,
        "screen": screen,
        "source_file": str(fpath),
        "widgets": widgets_out,
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

total_widgets = sum(len(item["widgets"]) for item in result)
print(f"Файлов обработано: {len(result)}")
print(f"Виджетов всего: {total_widgets}")
print(f"Сохранено: {OUTPUT_JSON.resolve()}")

# Этап 2: JSON -> брифы через LLM по API

Формат брифа - строка на показатель:

```
**Сборы** 1 588 млн руб., выполнение плана {color:ColorWarningRed}99,7%{color}
Кол-во клиентов 100 тыс. чел. ({color:ColorWarningGreen}+14,5%{color} vs 07M26), выполнение плана {color:ColorWarningRed}89,7%{color}
```

Отбираются 5-8 показателей с наибольшим отклонением (от плана и/или от предыдущего периода).
Дополнительно собирается короткая версия (1-2 самых важных показателя из полного брифа).

**Разделение труда:**
- **Python** считает сами цифры (значение, % к плану, % к предыдущему периоду) и отбирает топ-показатели - здесь ошибиться нельзя, поэтому без LLM.
- **Python** также решает цвет "выполнение плана" по единому правилу: ≥100% - зелёный, <100% - красный.
- **LLM** решает только то, что требует смысловой оценки: зелёный/красный для тренда к предыдущему периоду (для одних метрик рост - это хорошо, для других плохо), что выделить жирным, и собирает финальный текст строго по формату.

In [ ]:
import requests
from collections import defaultdict

# ================== НАСТРОЙКИ ЭТАПА 2 ==================
API_BASE = "http://your-endpoint/v1"    # ЗАПОЛНИТЕ: базовый URL API (без /chat/completions в конце)
API_KEY = ""                             # ЗАПОЛНИТЕ, если нужен токен
MODEL_NAME = "your-model"                # ЗАПОЛНИТЕ: точное имя модели

REPORT_PERIOD = "08.2026"  # <- параметр: срез "по состоянию на", формат "MM.ГГГГ" или "Nкв.ГГГГ",
                             # должен совпадать с форматом поля "period" в combined_widgets.json
TOP_N_MIN = 5
TOP_N_MAX = 8

INPUT_JSON = Path("combined_widgets.json")
OUTPUT_BRIEFS_JSON = Path("briefs.json")
PROMPTS_DIR = Path("prompts")  # сюда сохраняется промпт по каждому Дэшу отдельным txt-файлом
RESPONSES_DIR = Path("responses")  # сюда кладите ответ модели вручную, если прогоняете не через API

COLOR_RED_OPEN = "{color:ColorWarningRed}"
COLOR_RED_CLOSE = "{color}"
COLOR_GREEN_OPEN = "{color:ColorWarningGreen}"
COLOR_GREEN_CLOSE = "{color}"


def call_llm(prompt: str) -> str:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3,
        "max_tokens": 1500,
    }
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    resp = requests.post(f"{API_BASE}/chat/completions", json=payload, headers=headers, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    try:
        message = data["choices"][0]["message"]
    except (KeyError, IndexError, TypeError) as e:
        raise ValueError(
            "Неожиданная структура ответа от API (нет choices[0].message). Полный ответ:\n"
            + json.dumps(data, ensure_ascii=False, indent=2)[:2000]
        ) from e

    content = message.get("content")
    if content is None:
        alt = message.get("reasoning_content")
        hint = (
            "Похоже, текст лежит в поле 'reasoning_content' (это reasoning-модель) - "
            "нужно явно забирать его вместо/вместе с 'content'."
            if alt else
            "Проверьте: не сработал ли content-фильтр, finish_reason в ответе, "
            "правильно ли собран запрос под вашу модель (см. finish_reason и usage ниже)."
        )
        raise ValueError(
            "Модель вернула пустой content (None). Полное сообщение от API:\n"
            + json.dumps(message, ensure_ascii=False, indent=2)[:2000]
            + f"\n\nfinish_reason: {data['choices'][0].get('finish_reason')}"
            + f"\n{hint}"
        )

    return content

## Тест подключения к LLM

Запустите эту ячейку отдельно, прежде чем гонять весь пайплайн - проверяет, что `API_BASE`/`API_KEY`/`MODEL_NAME` настроены верно, без лишних действий.

In [ ]:
print("=" * 70)
print("ТЕСТ ПОДКЛЮЧЕНИЯ К LLM")
print("=" * 70)
print(f"API_BASE:   {API_BASE}")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"API_KEY:    {'задан' if API_KEY else 'НЕ задан'}")
print()

try:
    test_answer = call_llm("Ответь ровно одним словом: OK")
    print("Подключение работает.")
    print("Ответ модели:", repr(test_answer))
except requests.exceptions.ConnectionError as e:
    print("Не удалось подключиться к API_BASE - проверьте адрес/VPN/доступность сети изнутри этой машины.")
    print("Подробности:", e)
    raise
except requests.exceptions.HTTPError as e:
    print(f"Сервер ответил ошибкой {e.response.status_code}.")
    print("Тело ответа:", e.response.text[:1000])
    print("Проверьте: правильный ли MODEL_NAME (как он зарегистрирован на сервере), нужен ли API_KEY, "
          "правильный ли путь (некоторые серверы ждут не /v1/chat/completions, а другой путь).")
    raise
except Exception as e:
    print("Неожиданная ошибка:", repr(e))
    raise

## Расчёт цифр и отбор топ-показателей (Python, без LLM)

In [ ]:
def format_number_ru(value, decimals=0):
    if value is None:
        return "н/д"
    formatted = f"{value:,.{decimals}f}"
    integer_part, _, frac_part = formatted.partition(".")
    integer_part = integer_part.replace(",", " ")
    return f"{integer_part},{frac_part}" if decimals > 0 else integer_part


def format_pct_ru(value, decimals=1, with_sign=False):
    if value is None:
        return None
    sign = "+" if (with_sign and value > 0) else ""
    formatted = f"{sign}{value:.{decimals}f}".replace(".", ",")
    return f"{formatted}%"


def format_period_brief(period_str, granularity):
    # '08.2026' -> '08M26'; '3кв.2026' -> '3Q26'
    if granularity == "Месяц":
        mm, yyyy = period_str.split(".")
        return f"{mm}M{yyyy[-2:]}"
    num, yyyy = period_str.split("кв.")
    return f"{num}Q{yyyy[-2:]}"


def compute_metric_summary(widget, report_period):
    values = widget["values"]
    idx = next((i for i, v in enumerate(values) if v["period"] == report_period), None)
    if idx is None:
        return None

    current = values[idx]
    prev = values[idx - 1] if idx > 0 else None
    fact = current["fact"]
    plan = current["plan"]
    if fact is None:
        return None

    delta_pct = None
    if prev and prev["fact"] not in (None, 0):
        delta_pct = (fact - prev["fact"]) / abs(prev["fact"]) * 100

    plan_completion_pct = None
    if plan not in (None, 0):
        plan_completion_pct = fact / plan * 100

    deviation_score = 0.0
    if delta_pct is not None:
        deviation_score = max(deviation_score, abs(delta_pct))
    if plan_completion_pct is not None:
        deviation_score = max(deviation_score, abs(100 - plan_completion_pct))

    return {
        "title": widget["title"],
        "unit": widget.get("unit") or "",
        "granularity": widget["granularity"],
        "fact": fact,
        "prev_period": prev["period"] if prev else None,
        "delta_pct": delta_pct,
        "plan_completion_pct": plan_completion_pct,
        "deviation_score": deviation_score,
    }


def select_top_metrics(dash_items, report_period):
    summaries = []
    for item in dash_items:
        for w in item["widgets"]:
            s = compute_metric_summary(w, report_period)
            if s:
                summaries.append(s)
    summaries.sort(key=lambda s: s["deviation_score"], reverse=True)
    return summaries[:TOP_N_MAX]


def plan_color_tag(pct):
    if pct is None:
        return None
    return "green" if pct >= 100 else "red"


def prepare_metric_for_prompt(s):
    delta_display = None
    if s["delta_pct"] is not None:
        prev_disp = format_period_brief(s["prev_period"], s["granularity"])
        delta_display = f"{format_pct_ru(s['delta_pct'], with_sign=True)} vs {prev_disp}"

    plan_display = format_pct_ru(s["plan_completion_pct"]) if s["plan_completion_pct"] is not None else None

    return {
        "title": s["title"],
        "value_display": f"{format_number_ru(s['fact'])} {s['unit']}".strip(),
        "delta_display": delta_display,
        "plan_completion_display": plan_display,
        "plan_completion_color": plan_color_tag(s["plan_completion_pct"]),
    }

## Сборка промпта и вызов LLM

In [ ]:
PROMPT_TEMPLATE = """Собери бриф по дашборду "{dash}" по состоянию на {report_period_brief}.

Ниже - готовые показатели (Python уже посчитал значения и % - НЕ пересчитывай и не меняй цифры).
Для каждого показателя дано:
title - название
value_display - значение с единицей измерения
delta_display - строка вида "+14,5% vs 07M26" (изменение к предыдущему периоду) или null, если недоступно
plan_completion_display - % выполнения плана или null
plan_completion_color - "green" или "red" - ЭТОТ ЦВЕТ УЖЕ РЕШЁН, используй как есть, не меняй

Данные (JSON):
{metrics_json}

Собери текст СТРОГО по этому формату (по одной строке на показатель):

**Название1** value_display1, выполнение плана {red_open}XX,X%{red_close}
Название2 value_display2 ({green_open}+X,X% vs MMM YY{green_close}), выполнение плана {red_open}XX,X%{red_close}

Правила:
- Если delta_display есть - добавь его в скобках после value_display: "(delta_display)". Если delta_display = null - скобки не добавляй вообще.
- Цвет для delta_display определи САМ по смыслу показателя: рост - это хорошо или плохо для конкретной метрики (например, рост выручки/клиентов - обычно хорошо ({green_open}...{green_close}), рост затрат/оттока - обычно плохо ({red_open}...{red_close})). Если сомневаешься - считай рост позитивным.
- Открывающий/закрывающий тег для зелёного: {green_open} ... {green_close}
- Открывающий/закрывающий тег для красного: {red_open} ... {red_close}
- Если plan_completion_display = null - не добавляй фразу "выполнение плана" для этого показателя вообще.
- Цвет для "выполнение плана" бери ИЗ plan_completion_color как есть (green -> {green_open}...{green_close}, red -> {red_open}...{red_close}) - сам не решай.
- Название показателя оберни в ** (жирный) только для 1-2 самых критичных строк (наибольшее отклонение), остальные - обычным текстом.
- Пиши по-русски, без дополнительных пояснений и markdown-заголовков - только сами строки брифа.

Дополнительно собери КОРОТКУЮ версию - те же 1-2 самые критичные строки (максимальное отклонение), в том же формате.

Ответь СТРОГО в виде JSON без пояснений и без markdown-ограждения, в виде:
{{"full": "...полный бриф, строки через \n...", "short": "...короткий бриф..."}}
"""


def build_prompt(dash_name, dash_items):
    summaries = select_top_metrics(dash_items, REPORT_PERIOD)
    metrics = [prepare_metric_for_prompt(s) for s in summaries]
    metrics_json = json.dumps(metrics, ensure_ascii=False, indent=2)

    # гранулярность самого REPORT_PERIOD определяем по его собственному виду ("кв." или нет),
    # а не по гранулярности произвольного виджета - иначе падает на дэшах с квартальными
    # показателями при месячном REPORT_PERIOD (и наоборот)
    report_period_granularity = "Квартал" if "кв." in REPORT_PERIOD else "Месяц"
    report_period_brief = format_period_brief(REPORT_PERIOD, report_period_granularity)

    return PROMPT_TEMPLATE.format(
        dash=dash_name,
        report_period_brief=report_period_brief,
        metrics_json=metrics_json,
        red_open=COLOR_RED_OPEN, red_close=COLOR_RED_CLOSE,
        green_open=COLOR_GREEN_OPEN, green_close=COLOR_GREEN_CLOSE,
    )


def extract_json_from_text(text):
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1 and end > start:
            text = text[start:end + 1]
    return json.loads(text)

## Запуск по всем Дэшам

In [ ]:
def load_combined(path: Path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def group_by_dash(items):
    grouped = defaultdict(list)
    for item in items:
        grouped[item["dash"]].append(item)
    return grouped


def safe_filename(name: str) -> str:
    return re.sub(r'[\\/*?:"<>|]', "_", str(name).strip())


PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

items = load_combined(INPUT_JSON)
grouped = group_by_dash(items)
print(f"Дэшей: {len(grouped)} -> {list(grouped.keys())}")

briefs = {}
for dash_name, dash_items in grouped.items():
    prompt = build_prompt(dash_name, dash_items)

    prompt_path = PROMPTS_DIR / f"{safe_filename(dash_name)}.txt"
    prompt_path.write_text(prompt, encoding="utf-8")
    print(f"Промпт сохранён: {prompt_path}")

    print(f"Отправляю бриф по дэшу: {dash_name}...")
    try:
        raw = call_llm(prompt)
        parsed = extract_json_from_text(raw)
        briefs[dash_name] = parsed
    except Exception as e:
        print(f"  ⚠️ Через API не получилось ({e}) - промпт всё равно сохранён в {prompt_path}, "
              f"можно прогнать вручную и положить ответ в responses/{safe_filename(dash_name)}.json "
              f"(см. следующую ячейку)")
        briefs[dash_name] = None

with open(OUTPUT_BRIEFS_JSON, "w", encoding="utf-8") as f:
    json.dump(briefs, f, ensure_ascii=False, indent=2)

print(f"\nСохранено: {OUTPUT_BRIEFS_JSON.resolve()}")
for dash_name, brief in briefs.items():
    print(f"\n===== {dash_name} =====")
    if brief:
        print("--- Полный ---")
        print(brief.get("full"))
        print("--- Короткий ---")
        print(brief.get("short"))
    else:
        print("(ошибка)")

## (Опционально) Подхват ответов, прогнанных вручную

Если по какому-то Дэшу API не сработал - откройте `prompts/<Дэш>.txt`, вставьте в чат вашей LLM,
ответ модели (тот самый JSON `{"full": ..., "short": ...}`) сохраните как `responses/<Дэш>.json`
и запустите эту ячейку - она подхватит такие файлы и обновит `briefs.json`.

In [ ]:
RESPONSES_DIR.mkdir(parents=True, exist_ok=True)

picked_up = 0
for dash_name in grouped.keys():
    response_path = RESPONSES_DIR / f"{safe_filename(dash_name)}.json"
    if not response_path.exists():
        continue
    try:
        manual_text = response_path.read_text(encoding="utf-8")
        briefs[dash_name] = extract_json_from_text(manual_text)
        picked_up += 1
        print(f"Подхвачен ручной ответ: {dash_name} <- {response_path}")
    except Exception as e:
        print(f"⚠️ Не удалось разобрать {response_path}: {e}")

if picked_up:
    with open(OUTPUT_BRIEFS_JSON, "w", encoding="utf-8") as f:
        json.dump(briefs, f, ensure_ascii=False, indent=2)
    print(f"\nОбновлено {picked_up} брифов(а), сохранено в {OUTPUT_BRIEFS_JSON.resolve()}")
else:
    print("Файлов с ручными ответами не найдено в", RESPONSES_DIR.resolve())